# Stage 5. Object Segmentation Pipeline

## Notebook Information

**Purpose:**  
Segment cellular objects in background-corrected field-of-view images and save segmentation masks.

**Workflow summary:**  
Load segmentation parameters, Part 4b metadata, and background-corrected images; select the training subset; preprocess nucleus, concanavalin, and actin channels; run Cellpose segmentation; then save segmentation masks and updated metadata.

**Run scope:**  
Training subset by default, using rows where `is_train == 1`.

**Reproducibility:**  
Use the configuration/hyperparameter cell as the source of truth for paths, channel positions, preprocessing parameters, Cellpose parameters, output dtype, and save settings. Run sections in order for a fresh execution.

### Authors

| Name | Affiliation |
|---|---|
| Alessandro Ulivi | Infectious Diseases Imaging Platform, Center for Integrative Infectious Disease Research, Heidelberg |
| Edwin Carreno | Scientific Software Center, Heidelberg |
| Christine Schultz | Scientific Software Center, Heidelberg |
| Name Surname | Infectious Diseases Imaging Platform, Center for Integrative Infectious Disease Research, Heidelberg |

## 1. Setup

### 1.1 Imports

In [ ]:
# Built-in imports
import datetime
import logging
import os
from importlib.metadata import version
from pathlib import Path

# Third-party imports
import numpy as np
import pandas as pd
from omegaconf import OmegaConf
from skimage.transform import resize

# Package imports
from acid.utils.listdirNHF import listdirNHF
from acid.utils.get_defaults import default_file_name
from acid.utils.save_image import tifffile_save_ometiff
from acid.image_processing.extract_metadata import extract_ometif_imagej_metadata
from acid.image_processing.make_imagej_metadata import imagej_compatible_metadata_dict
from acid.utils.fov_axis_utils import get_fov_ch_shape

# Package imports NEW
from acid.utils.filesystem.filesystem import create_output_directories
from acid.utils.metadata.filtering import filter_metadata_by_splits
from acid.utils.metadata.loading import load_metadata
from acid.segmentation.registry import create_segmentation_model

# ----- Commented in original notebook
# import tifffile
# import matplotlib.pyplot as plt

# ----- Temporal (delete once refactored)

### 1.2 Configure logging

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

### 1.3 Load configuration

In [ ]:
CONFIG_PATH_FILE = Path("config.yaml")

if not CONFIG_PATH_FILE.is_file():
    CONFIG_PATH_FILE = Path("notebooks_refactored/config.yaml")

CONFIG_PATH_FILE = CONFIG_PATH_FILE.resolve()
config = OmegaConf.load(CONFIG_PATH_FILE)
for key, value in config.shared.paths.items():
    config.shared.paths[key] = str((CONFIG_PATH_FILE.parent / value).resolve())

In [ ]:
paths_cfg = config.shared.paths

# Metadata config
metadata_cfg = config.object_segmentation.metadata

# Dataset split config
dataset_split_cfg = config.object_segmentation.dataset_split

# Processing config
segmentation_processing_cfg = config.object_segmentation.processing
segmentation_processing_cfg

## 2. Input Data Preparation

### 2.1 Create output directories

In [ ]:
output_path, _ = create_output_directories(
    output_directory=paths_cfg.segmentation_masks_dir,
    enable_secondary_output=False,
)

### 2.2 Load metadata dataframe

In [ ]:
metadata_df, metadata_file_name = load_metadata(metadata_config=metadata_cfg)

In [ ]:
metadata_file_name

In [ ]:
metadata_df.head()

### 2.3 Select training images

In [ ]:
metadata_df = filter_metadata_by_splits(
    metadata_df=metadata_df, selected_splits="train", split_config=dataset_split_cfg
)


### 2.4 Get the shape and the number of channels of the fields of view

In [ ]:
fov_shape, num_channels, shape_of_channels = get_fov_ch_shape(
    df=metadata_df,
    fov_dir=paths_cfg.corrected_fov_dir,
    fov_clm=metadata_cfg.dataframe_columns.illum_correct_df_file_name_clm_name,
    channel_axis=segmentation_processing_cfg.channel_axis,
    null_value=metadata_cfg.dataframe_columns.null_value,
)

## 3. Instantiate Segmentation model

### 3.1 Import CellPose and CellPose models

In [ ]:
model = create_segmentation_model(
    name=segmentation_processing_cfg.model_backend,
)

logging.info(f"Model name: {model.name}")
logging.info(f"Model version: {model.version}")

## 4. Cellular Object detection


- 5.1. Preprocess field of views.

- 5.2. Segment cell and nucleus.

- 5.2. Save segmentation masks

- 5.3. Update and save metadata

In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
from skimage.transform import resize
from tqdm.notebook import tqdm

from acid.image_processing.extract_metadata import extract_ometif_imagej_metadata
from acid.image_processing.filter_image import median_filter_image
from acid.image_processing.make_imagej_metadata import imagej_compatible_metadata_dict
from acid.image_processing.resize_image import downsample_local_mean
from acid.utils.save_image import tifffile_save_ometiff

##### 4.1. Preprocess field of views

In [ ]:
# FUTURE: move function to src/acid/segmentation/io.py
def get_field_of_view_file(metadata_row, config) -> str:
    field_of_view_file = metadata_row.get(
        config.metadata.dataframe_columns.illum_correct_df_file_name_clm_name
    )

    if pd.isna(field_of_view_file) or str(field_of_view_file).strip() == "":
        raise ValueError(
            f"Missing FOV filename in column {config.metadata.dataframe_columns.illum_correct_df_file_name_clm_name!r}"
        )

    return str(field_of_view_file).strip()


# FUTURE: move function to src/acid/segmentation/io.py
def load_field_of_view(field_of_view_file, corrected_fov_directory):
    field_of_view_path = Path(corrected_fov_directory) / field_of_view_file

    try:
        return tifffile.imread(field_of_view_path)
    except Exception as error:
        raise OSError(
            f"Could not load field of view TIFF: {field_of_view_path}"
        ) from error


# FUTURE: move function to src/acid/image_processing/segmentation_preprocessing.py
def select_segmentation_channels(image, config):
    unstacked_image = np.moveaxis(image, config.channel_axis, 0)

    return (
        unstacked_image[config.nucleus_position],
        unstacked_image[config.concanavalin_position],
        unstacked_image[config.actin_position],
    )


# FUTURE: move function to src/acid/image_processing/segmentation_preprocessing.py
def merge_concanavalin_actin_channels(concanavalin_channel, actin_channel):
    return np.mean(np.stack([concanavalin_channel, actin_channel], axis=0), axis=0)


# FUTURE: move function to src/acid/image_processing/segmentation_preprocessing.py
def preprocess_image_for_segmentation(image, config):
    nucleus_channel, concanavalin_channel, actin_channel = select_segmentation_channels(
        image=image,
        config=config,
    )

    concactin_merge = merge_concanavalin_actin_channels(
        concanavalin_channel=concanavalin_channel,
        actin_channel=actin_channel,
    )

    med_nucleus = median_filter_image(
        nucleus_channel,
        size=config.med_filter_nucleus,
    )

    med_concactin = median_filter_image(
        concactin_merge,
        size=config.med_filter_concactin_merge,
    )

    restacked_image = np.stack(
        [med_nucleus, med_concactin],
        axis=config.channel_axis,
    )

    return downsample_local_mean(
        restacked_image,
        factor=config.downsampling_factor,
        channel_axis=config.channel_axis,
    )

##### 4.2. Segment cell and nucleus

In [ ]:
# FUTURE: move function to src/acid/segmentation/apply_segmentation.py
def segment_objects(preprocessed_image, model, config):
    return model.eval(
        preprocessed_image,
        flow_threshold=config.processing.flow_threshold,
        cellprob_threshold=config.processing.cellprob_threshold,
        diameter=config.processing.diameter,
        channel_axis=config.processing.channel_axis,
    )


# FUTURE: move function to src/acid/segmentation/apply_segmentation.py
def resize_segmentation_mask(mask, output_shape, config):
    resized_mask = resize(
        mask,
        output_shape=output_shape,
        order=config.processing.order,
        preserve_range=config.processing.preserve_range,
        anti_aliasing=config.processing.anti_aliasing,
    )

    return resized_mask.astype(mask.dtype, copy=False)

##### 4.3. Save segmentation masks

In [ ]:
# FUTURE: move function to src/acid/segmentation/metadata.py
def copy_selected_field_of_view_metadata(
    segmentation_metadata, field_of_view_metadata, config
):
    metadata_keywords = [
        config.metadata.image_metadata.preproc_img_meta_raw_file_name_entry,
        config.metadata.image_metadata.preproc_img_meta_scene_file_name_entry,
        config.metadata.image_metadata.preproc_img_meta_x_physic_px_size_entry,
        config.metadata.image_metadata.preproc_img_meta_y_physic_px_size_entry,
        config.metadata.image_metadata.preproc_img_meta_x_physic_px_size_unit_entry,
        config.metadata.image_metadata.preproc_img_meta_y_physic_px_size_unit_entry,
    ]

    for key, value in field_of_view_metadata.items():
        if any(keyword in key for keyword in metadata_keywords):
            segmentation_metadata[key] = value

    return segmentation_metadata


# FUTURE: move function to src/acid/segmentation/metadata.py
def build_segmentation_image_metadata(field_of_view_path, mask, model, config):
    field_of_view_metadata = extract_ometif_imagej_metadata(field_of_view_path)

    processing_steps = (
        config.metadata.image_metadata.segmented_img_meta_processing_steps
    )

    if config.processing.output_dtype is not None:
        processing_steps = f"{processing_steps} change output data type to {config.processing.output_dtype} for saving."

    segmentation_metadata = {
        config.metadata.image_metadata.segmented_img_meta_date_name: datetime.now().strftime(
            config.metadata.image_metadata.processing_date_format
        ),
        config.metadata.image_metadata.segmented_img_meta_method_name: config.metadata.segmentation_method_name,
        config.metadata.image_metadata.segmentation_method_version_name: getattr(
            model, "version", None
        ),
        config.metadata.image_metadata.segmented_img_meta_diameter_name: config.processing.diameter,
        config.metadata.image_metadata.segmented_img_meta_flow_threshold_name: config.processing.flow_threshold,
        config.metadata.image_metadata.segmented_img_meta_cellprob_threshold_name: config.processing.cellprob_threshold,
        config.metadata.image_metadata.segmented_img_meta_downsampling_factor_name: config.processing.downsampling_factor,
        config.metadata.image_metadata.segmented_img_meta_nucleus_med_filter_size_name: config.processing.med_filter_nucleus,
        config.metadata.image_metadata.segmented_img_meta_concactin_merge_med_filter_size_name: (
            config.processing.med_filter_concactin_merge
        ),
        config.metadata.image_metadata.segmented_img_meta_resize_order_name: config.processing.order,
        config.metadata.image_metadata.segmented_img_meta_processing_name: processing_steps,
        config.metadata.image_metadata.segmented_img_meta_dtype_name: str(mask.dtype),
    }

    imagej_metadata = imagej_compatible_metadata_dict(segmentation_metadata)

    return copy_selected_field_of_view_metadata(
        segmentation_metadata=imagej_metadata,
        field_of_view_metadata=field_of_view_metadata,
        config=config,
    )


# FUTURE: move function to src/acid/segmentation/io.py
def make_segmentation_output_filename(field_of_view_file, config):
    field_of_view_file = Path(field_of_view_file)
    suffix = config.image_saving.ome_suffix
    stem = field_of_view_file.name.removesuffix(suffix)

    return (
        f"{stem}"
        f"{config.image_saving.save_file_name_separator}"
        f"{config.image_saving.segmentation_savingword}"
        f"{suffix}"
    )


# FUTURE: move function to src/acid/segmentation/io.py
def save_segmentation_mask(
    output_filename, mask, image_metadata, config, output_directory
):
    output_path = Path(output_directory) / output_filename
    output_path.parent.mkdir(parents=True, exist_ok=True)

    tifffile_save_ometiff(
        output_path,
        data=mask,
        imagej=config.image_saving.save_imagej_compatible,
        photometric=config.image_saving.photometric,
        metadata=image_metadata,
    )

    return output_path

In [ ]:
config.object_segmentation.image_saving.ome_suffix

##### 4.4. Update and save metadata

In [ ]:
# FUTURE: move function to src/acid/segmentation/metadata.py
def get_segmentation_metadata_columns(config) -> list[str]:
    return [
        config.metadata.dataframe_columns.metadata_df_date_clm_name,
        config.metadata.dataframe_columns.metadata_df_file_name_clm_name,
        config.metadata.dataframe_columns.metadata_df_method_clm_name,
        config.metadata.dataframe_columns.metadata_df_method_version_clm_name,
        config.metadata.dataframe_columns.metadata_df_diameter_clm_name,
        config.metadata.dataframe_columns.metadata_df_flow_threshold_clm_name,
        config.metadata.dataframe_columns.metadata_df_cellprob_threshold_clm_name,
        config.metadata.dataframe_columns.metadata_df_downsampling_factor_clm_name,
        config.metadata.dataframe_columns.metadata_df_nucleus_med_filter_size_name,
        config.metadata.dataframe_columns.metadata_df_concactin_merge_med_filter_size_name,
        config.metadata.dataframe_columns.metadata_df_resize_order_name,
        config.metadata.dataframe_columns.metadata_df_output_dtype_name,
    ]


# FUTURE: move function to src/acid/segmentation/metadata.py
def make_segmentation_success_result(
    row_index, input_file, output_file, mask_dtype, model, config
):
    return {
        "row_index": row_index,
        "input_file": input_file,
        "output_file": output_file,
        "success": True,
        "stage": None,
        "error_type": None,
        "error_message": None,
        config.metadata.dataframe_columns.metadata_df_date_clm_name: datetime.now().strftime(
            config.metadata.dataframe_columns.metadata_df_meta_date_format
        ),
        config.metadata.dataframe_columns.metadata_df_file_name_clm_name: output_file,
        config.metadata.dataframe_columns.metadata_df_method_clm_name: config.metadata.segmentation_method_name,
        config.metadata.dataframe_columns.metadata_df_method_version_clm_name: getattr(
            model, "version", None
        ),
        config.metadata.dataframe_columns.metadata_df_diameter_clm_name: config.processing.diameter,
        config.metadata.dataframe_columns.metadata_df_flow_threshold_clm_name: config.processing.flow_threshold,
        config.metadata.dataframe_columns.metadata_df_cellprob_threshold_clm_name: config.processing.cellprob_threshold,
        config.metadata.dataframe_columns.metadata_df_downsampling_factor_clm_name: config.processing.downsampling_factor,
        config.metadata.dataframe_columns.metadata_df_nucleus_med_filter_size_name: config.processing.med_filter_nucleus,
        config.metadata.dataframe_columns.metadata_df_concactin_merge_med_filter_size_name: (
            config.processing.med_filter_concactin_merge
        ),
        config.metadata.dataframe_columns.metadata_df_resize_order_name: config.processing.order,
        config.metadata.dataframe_columns.metadata_df_output_dtype_name: str(
            mask_dtype
        ),
    }


# FUTURE: move function to src/acid/segmentation/metadata.py
def make_segmentation_failure_result(row_index, input_file, error, config, stage=None):
    result = {
        "row_index": row_index,
        "input_file": input_file,
        "output_file": None,
        "success": False,
        "stage": stage,
        "error_type": type(error).__name__,
        "error_message": str(error),
    }

    for column in get_segmentation_metadata_columns(config):
        result[column] = config.metadata.dataframe_columns.null_value

    return result


# FUTURE: move function to src/acid/segmentation/metadata.py
def update_metadata_with_segmentation_results(
    metadata_df,
    results,
    config,
    copy_dataframe=True,
):
    if copy_dataframe:
        metadata_df = metadata_df.copy()

    if not results:
        return metadata_df

    metadata_columns = get_segmentation_metadata_columns(config)
    results_df = pd.DataFrame.from_records(results).set_index("row_index")

    missing_result_columns = [
        column for column in metadata_columns if column not in results_df.columns
    ]

    if missing_result_columns:
        raise KeyError(
            f"Segmentation results are missing metadata columns: {missing_result_columns}"
        )

    missing_metadata_columns = [
        column for column in metadata_columns if column not in metadata_df.columns
    ]

    metadata_df = metadata_df.assign(
        **{column: pd.NA for column in missing_metadata_columns}
    )

    metadata_df = metadata_df.astype(dict.fromkeys(metadata_columns, "object"))
    metadata_df.loc[results_df.index, metadata_columns] = results_df[
        metadata_columns
    ].to_numpy()

    return metadata_df

In [ ]:
def get_channel_shape(image, channel_axis):
    """Return the image shape excluding its channel axis."""
    return tuple(size for axis, size in enumerate(image.shape) if axis != channel_axis)

In [ ]:
def cast_mask_to_output_dtype(mask, output_dtype):
    """Cast a segmentation mask to the configured output dtype, when provided."""
    if output_dtype is None:
        return mask

    return mask.astype(output_dtype)

##### 4.5 Segmentation processing in batch

In [ ]:
# FUTURE: move function to src/acid/segmentation/apply_segmentation.py
def apply_segmentation_for_fov(row_index, metadata_row, model, config, paths):

    # 1. Extract the filename from the row
    try:
        field_of_view_file = get_field_of_view_file(metadata_row, config=config)

    except Exception as error:
        return make_segmentation_failure_result(
            row_index=row_index,
            input_file=None,
            error=error,
            config=config,
            stage="get_field_of_view_file",
        )

    # 2. Load the field of view with background corrected.
    try:
        image = load_field_of_view(field_of_view_file, paths.corrected_fov_dir)
    except Exception as error:
        return make_segmentation_failure_result(
            row_index=row_index,
            input_file=field_of_view_file,
            error=error,
            config=config,
            stage="load_field_of_view",
        )

    # 3. Preprocess the image for segmentation
    try:
        preprocessed_image = preprocess_image_for_segmentation(
            image=image, config=config.processing
        )

    except Exception as error:
        return make_segmentation_failure_result(
            row_index=row_index,
            input_file=field_of_view_file,
            error=error,
            config=config,
            stage="preprocess_image_for_segmentation",
        )
    logging.debug(f"Preprocessing image shape: {preprocessed_image.shape}")

    # 4. Segment objects
    try:
        masks, flows, styles = segment_objects(
            preprocessed_image=preprocessed_image, model=model, config=config
        )
    except Exception as error:
        return make_segmentation_failure_result(
            row_index=row_index,
            input_file=field_of_view_file,
            error=error,
            config=config,
            stage="segment_preprocessed_image",
        )
    logging.debug(f"Masks: {masks.shape}")
    num_objects = np.unique(masks).size - 1
    logging.debug(f"Number of segmented objects: {num_objects}")
    logging.debug(f"Data type mask: {masks.dtype}")

    # 5. Get channel shape
    channel_shape = get_channel_shape(
        image=image,
        channel_axis=config.processing.channel_axis,
    )
    logging.info(f"Channel shape: {channel_shape}")

    # 6. Resize segmentation mask
    try:
        mask = resize_segmentation_mask(
            mask=masks, output_shape=channel_shape, config=config
        )
    except Exception as error:
        return make_segmentation_failure_result(
            row_index=row_index,
            input_file=field_of_view_file,
            error=error,
            config=config,
            stage="resize_segmentation_mask",
        )
    logging.debug(f"Resize mask: {mask.shape}")

    # 7. Cast mask to output type
    mask = cast_mask_to_output_dtype(
        mask=mask,
        output_dtype=config.processing.output_dtype,
    )

    # 8. Build segmentation image metadata
    try:
        field_of_view_path = Path(paths.corrected_fov_dir) / field_of_view_file
        image_metadata = build_segmentation_image_metadata(
            field_of_view_path=field_of_view_path,
            mask=mask,
            model=model,
            config=config,
        )
    except Exception as error:
        return make_segmentation_failure_result(
            row_index=row_index,
            input_file=field_of_view_file,
            error=error,
            config=config,
            stage="build_segmentation_image_metadata",
        )
    logging.debug(f"Image metadata: {image_metadata}")

    # 9. save segmentation masks
    try:
        output_filename = make_segmentation_output_filename(field_of_view_file, config)
        logging.debug(f"Output file: {output_filename}")

        output_path = save_segmentation_mask(
            output_filename=output_filename,
            mask=mask,
            image_metadata=image_metadata,
            config=config,
            output_directory=paths.segmentation_masks_dir,
        )
    except Exception as error:
        logging.error("Error saving masks")
        return make_segmentation_failure_result(
            row_index=row_index,
            input_file=field_of_view_file,
            error=error,
            config=config,
            stage="save_segmentation_mask",
        )
    logging.info(f"Output path: {output_filename}")

    return make_segmentation_success_result(
        row_index=row_index,
        input_file=field_of_view_file,
        output_file=output_filename,
        mask_dtype=mask.dtype,
        model=model,
        config=config,
    )


# FUTURE: move function to src/acid/segmentation/apply_segmentation.py
def apply_segmentation_batch(metadata_df, model, config, paths, max_rows=None):
    row_indices = metadata_df.index

    if max_rows is not None:
        row_indices = metadata_df.index[:max_rows]

    results = []

    for row_index in tqdm(row_indices, desc="Applying object segmentation"):
        result = apply_segmentation_for_fov(
            row_index=row_index,
            metadata_row=metadata_df.loc[row_index],
            model=model,
            config=config,
            paths=paths,
        )
        results.append(result)

    return results

In [ ]:
results = apply_segmentation_batch(
    metadata_df=metadata_df,
    model=model,
    config=config.object_segmentation,
    paths=paths_cfg,
)

results

In [ ]:
from pathlib import Path
from datetime import datetime


# FUTURE: move function to src/acid/utils/metadata/saving.py
def build_metadata_dataframe_filename(
    metadata_saving_config, project_name, timestamp=None
):
    """Build the standard ACID metadata dataframe filename."""
    if timestamp is None:
        timestamp = datetime.now()

    separator = metadata_saving_config.save_file_name_separator
    metadata_file_suffix = metadata_saving_config.metadata_file_suffix.format(
        save_file_name_separator=separator
    )

    return separator.join(
        [
            timestamp.strftime(metadata_saving_config.metadata_date_format),
            project_name,
            metadata_saving_config.metadata_savingword,
            metadata_file_suffix,
        ]
    )


def build_metadata_dataframe_path(metadata_config, project_name, timestamp=None):
    """Build the full save path for a metadata dataframe."""
    metadata_directory = metadata_config.directory
    metadata_filename = build_metadata_dataframe_filename(
        metadata_saving_config=metadata_config.saving,
        project_name=project_name,
        timestamp=timestamp,
    )

    return Path(metadata_directory) / metadata_filename


def save_metadata_dataframe(metadata_df, metadata_config, project_name, timestamp=None):
    """Save a metadata dataframe as CSV and return the saved path."""

    metadata_path = build_metadata_dataframe_path(
        metadata_config=metadata_config,
        project_name=project_name,
        timestamp=timestamp,
    )

    metadata_path.parent.mkdir(parents=True, exist_ok=True)

    metadata_df.to_csv(
        metadata_path,
        index=metadata_config.saving.save_csv_index,
    )

    return metadata_path

## 5. Update processing metadata dataframe

In [ ]:
metadata_df_updated = update_metadata_with_segmentation_results(
    metadata_df=metadata_df,
    results=results,
    config=config.object_segmentation,
    copy_dataframe=True,
)

metadata_df_updated.info()

## 6. Save metadata file

In [ ]:
metadata_path = save_metadata_dataframe(
    metadata_df=metadata_df_updated,
    metadata_config=metadata_cfg,
    project_name=config.project_identity.project_name,
)


logging.info(f"Saved metadata dataframe to: {metadata_path}")

## End of the notebook